# Dim Set — Gold Layer

Enriched set dimension joining `foundation_sets` with `dim_theme_hierarchy` and pre-aggregated inventory counts.

## What this notebook does

1. **Read** — Loads sets from silver and theme hierarchy from gold.
2. **Transform** — Joins sets with theme hierarchy and aggregates inventory-level metrics (unique colors, unique parts, minifig count).
3. **Write** — Overwrites the Delta table at `/Volumes/lego_brickbase/gold/delta_volume/dim_set`.
4. **Register** — Creates the catalog table `lego_brickbase.gold.dim_set`.

## Imports and Configuration

In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import col, countDistinct, sum as _sum, coalesce, lit


SILVER_SETS             = "lego_brickbase.silver.foundation_sets"
SILVER_INVENTORIES      = "lego_brickbase.silver.foundation_inventories"
SILVER_INVENTORY_PARTS  = "lego_brickbase.silver.foundation_inventory_parts"
SILVER_INVENTORY_MINIFIGS = "lego_brickbase.silver.foundation_inventory_minifigs"
GOLD_THEME_HIERARCHY    = "lego_brickbase.gold.dim_theme_hierarchy"
DELTA_TABLE_PATH        = "/Volumes/lego_brickbase/gold/delta_volume/dim_set"
CATALOG_TABLE           = "lego_brickbase.gold.dim_set"

## Read and Transform

Aggregate inventory-level metrics per set, then join with sets and theme hierarchy.

In [ ]:
df_sets = spark.table(SILVER_SETS)
df_themes = spark.table(GOLD_THEME_HIERARCHY)
df_inventories = spark.table(SILVER_INVENTORIES)
df_inv_parts = spark.table(SILVER_INVENTORY_PARTS)
df_inv_minifigs = spark.table(SILVER_INVENTORY_MINIFIGS)

# Aggregate parts metrics per set via inventories
df_parts_agg = (
    df_inventories
    .join(df_inv_parts, df_inventories["inventory_id"] == df_inv_parts["inventory_id"], "inner")
    .groupBy(df_inventories["set_key"])
    .agg(
        countDistinct(df_inv_parts["part_key"]).alias("number_of_unique_parts"),
        countDistinct(df_inv_parts["color_key"]).alias("number_of_unique_colors"),
        _sum(df_inv_parts["quantity"]).alias("total_part_quantity"),
    )
)

# Aggregate minifig count per set via inventories
df_minifig_agg = (
    df_inventories
    .join(df_inv_minifigs, df_inventories["inventory_id"] == df_inv_minifigs["inventory_id"], "inner")
    .groupBy(df_inventories["set_key"])
    .agg(
        _sum(df_inv_minifigs["quantity"]).alias("number_of_minifigs"),
    )
)

# Join everything together
df_dim = (
    df_sets
    .join(df_themes, df_sets["theme_key"] == df_themes["theme_key"], "left")
    .join(df_parts_agg, df_sets["set_key"] == df_parts_agg["set_key"], "left")
    .join(df_minifig_agg, df_sets["set_key"] == df_minifig_agg["set_key"], "left")
    .select(
        df_sets["set_key"],
        df_themes["theme_key"],
        df_sets["set_number"],
        df_sets["set_name"],
        df_sets["year"],
        df_themes["theme_name"],
        df_themes["parent_theme_name"],
        df_themes["root_theme_name"],
        df_sets["number_of_parts"],
        coalesce(col("number_of_unique_parts"), lit(0)).alias("number_of_unique_parts"),
        coalesce(col("number_of_unique_colors"), lit(0)).alias("number_of_unique_colors"),
        coalesce(col("total_part_quantity"), lit(0)).alias("total_part_quantity"),
        coalesce(col("number_of_minifigs"), lit(0)).alias("number_of_minifigs"),
        df_sets["set_image_url"],
        df_sets["set_url"],
        df_sets["source_last_modified_date"],
    )
)

df_dim.printSchema()
df_dim.display(10, truncate=False)

## Write to Delta Volume, Register Catalog Table, and Apply Constraints

In [ ]:
(
    df_dim
    .write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .save(DELTA_TABLE_PATH)
)

row_count = spark.read.format("delta").load(DELTA_TABLE_PATH).count()
print(f"Rows written to Delta table: {row_count:,}")

spark.sql(f"""
    CREATE OR REPLACE TABLE {CATALOG_TABLE} (
        set_key                   STRING    NOT NULL COMMENT 'Surrogate key for the set dimension',
        theme_key                 INTEGER            COMMENT 'Foreign key to the theme dimension (dim_theme_hierarchy)',
        set_number                STRING             COMMENT 'Official LEGO set number (e.g. 75192-1)',
        set_name                  STRING             COMMENT 'Official name of the LEGO set',
        year                      INTEGER            COMMENT 'Year the set was released',
        theme_name                STRING             COMMENT 'Name of the direct theme the set belongs to',
        parent_theme_name         STRING             COMMENT 'Name of the parent theme one level up in the hierarchy; NULL for top-level themes',
        root_theme_name           STRING             COMMENT 'Name of the root (top-level) theme in the hierarchy',
        number_of_parts           INTEGER            COMMENT 'Official part count as listed on the LEGO set packaging',
        number_of_unique_parts    BIGINT             COMMENT 'Count of distinct part designs included across all inventories of this set',
        number_of_unique_colors   BIGINT             COMMENT 'Count of distinct colors used across all parts in this set',
        total_part_quantity       BIGINT             COMMENT 'Total sum of all individual part quantities across all inventories',
        number_of_minifigs        BIGINT             COMMENT 'Total number of minifigures included in this set (summed across all inventories)',
        set_image_url             STRING             COMMENT 'URL to the primary product image of the set',
        set_url                   STRING             COMMENT 'URL to the set detail page on Rebrickable',
        source_last_modified_date TIMESTAMP          COMMENT 'Timestamp when the source record was last modified in the upstream system',
        CONSTRAINT dim_set_pk PRIMARY KEY (set_key)
    )
    COMMENT 'Every official LEGO set, including its name, release year, theme, and a summary of its contents — unique part count, color count, and number of minifigures.'
""")

spark.sql(f"""
    INSERT INTO {CATALOG_TABLE}
    SELECT * FROM delta.`{DELTA_TABLE_PATH}`
""")

print(f"Catalog table ready with constraints: {CATALOG_TABLE}")